# 栄養・レシピ

担当：高居
出力: `data/recipes.csv`（列は team_rules.md 第2節: recipe_id, name, ingredient, amount_g, cook_min, cleanup_min）
出力: `data/cooking_cost_monthly.csv`（月ごと・レシピごとの材料費）、`docs/cooking_cost_trend.png`（自炊費用の推移グラフ）

## 第4回
- レシピは全員で数品ずつ分担して、下の「レシピ入力」セルに同じ書式で追記する。書式の統一と検算は高居。
- `ingredient` は `prices.csv` の `name` と同じ文字列で出力する（対応表で変換）。
- 使用量は g にそろえる。個・mL は設定セルの換算値で g に直す。
- レシピはすべて分析用のオリジナル仮想レシピ（1人分）。出版物・Webサイトからの転載はしない。
- 栄養（成分表）は `nutrition.ipynb` で付ける。`amount_g` は「買う量（廃棄を含む）」として扱う。
- **push 前に「Clear All Outputs」を実行する**

> **このセル**：ファイルの場所と、gへの換算値（卵1個＝60g、牛乳1mL＝1g）を決めている。
>
> **聞かれたら**：換算値は仮定。あとで `config.py` に移す予定。

In [ ]:
# ===== 設定セル：ファイルの場所と、g への換算に使う仮の値 =====
from pathlib import Path           # ファイルの場所（パス）を扱う

import matplotlib.pyplot as plt    # グラフ用（日本語フォントの設定に使う）
import pandas as pd                # 表（DataFrame）を扱う

# ノートブックは src/ から実行する（optimize.ipynb と同じ。データは ../data/）。
PRICE_PATH = Path("../data/prices.csv")    # 入力：prices.ipynb が作った価格表
OUT_PATH = Path("../data/recipes.csv")     # 出力：完成したレシピ表（team_rules 第2節の列）
OUT_COST_PATH = Path("../data/cooking_cost_monthly.csv")  # 出力：月ごと・レシピごとの材料費（1人分・ロスなし）
OUT_FIG_PATH = Path("../docs/cooking_cost_trend.png")     # 出力：自炊費用の推移グラフ

# g への換算（仮定値。あとで src/config.py に移す）。価格や使用量を「重さ(g)」にそろえるために必要
EGG_G_PER_PIECE = 60   # 鶏卵 1個 = 60g（殻つきの目安として仮置き。可食部は廃棄率14%で約52g）
MILK_G_PER_ML = 1.0    # 1mL = 1g として扱う（牛乳。しょう油も同じ値で仮置き）

plt.rcParams["font.family"] = "Meiryo"  # Windows でグラフの日本語を表示する

## レシピ入力（全員でここに追記する）

1行 = 1レシピ × 1食材。`unit` は g / 個 / mL のどれか。
調理・片付け時間は `RECIPE_TIMES` にレシピごとに入れる。

> **このセル**：レシピの中身（料理名・食材・量・調理と片付けの時間）を書く場所。全員がここに追記する。
>
> **聞かれたら**：レシピは分析用に自分たちで作ったもので、転載ではない。時間はまだ仮の値。

In [ ]:
# ===== レシピ入力セル（全員がここに追記する。データの中身を書く場所） =====

# RAW_RECIPES：レシピの材料表。1行 = 1レシピ × 1食材（1人分）。
#   書式: (レシピID, 料理名, 食材の短い名前, 量, 単位)
#   食材の短い名前は、下の INGREDIENT_TO_ITEM に載っているものを使う。単位は "g" / "個" / "mL"。
#   レシピはすべて分析用のオリジナル仮想レシピ（転載ではない）。量は目安で、実測ではない。
RAW_RECIPES = [
    # recipe_id, name, ingredient（下の対応表の短い名前）, amount, unit
    ("R001", "鶏肉とキャベツのごはん", "うるち米", 150, "g"),
    ("R001", "鶏肉とキャベツのごはん", "鶏肉", 100, "g"),
    ("R001", "鶏肉とキャベツのごはん", "キャベツ", 80, "g"),
    ("R001", "鶏肉とキャベツのごはん", "鶏卵", 0.5, "個"),
    ("R002", "さけのミルクパスタ", "スパゲッティ", 100, "g"),
    ("R002", "さけのミルクパスタ", "さけ", 100, "g"),
    ("R002", "さけのミルクパスタ", "牛乳", 100, "mL"),
    ("R002", "さけのミルクパスタ", "キャベツ", 50, "g"),
    ("R003", "豚肉とじゃがいものみそ煮", "豚肉", 100, "g"),
    ("R003", "豚肉とじゃがいものみそ煮", "じゃがいも", 150, "g"),
    ("R003", "豚肉とじゃがいものみそ煮", "にんじん", 50, "g"),
    ("R003", "豚肉とじゃがいものみそ煮", "キャベツ", 50, "g"),
    ("R003", "豚肉とじゃがいものみそ煮", "鶏卵", 0.5, "個"),
    ("R003", "豚肉とじゃがいものみそ煮", "みそ", 15, "g"),  # 料理名に合わせて追加（量は要確認）
    ("R004", "鶏肉と野菜のうどん", "ゆでうどん", 200, "g"),
    ("R004", "鶏肉と野菜のうどん", "鶏肉", 80, "g"),
    ("R004", "鶏肉と野菜のうどん", "にんじん", 40, "g"),
    ("R004", "鶏肉と野菜のうどん", "キャベツ", 60, "g"),
    ("R005", "トースト朝食", "食パン", 120, "g"),
    ("R005", "トースト朝食", "鶏卵", 1, "個"),
    ("R005", "トースト朝食", "牛乳", 200, "mL"),
    ("R005", "トースト朝食", "トマト", 60, "g"),
    ("R006", "豆腐とねぎのみそ汁ごはん", "うるち米", 150, "g"),
    ("R006", "豆腐とねぎのみそ汁ごはん", "豆腐", 100, "g"),
    ("R006", "豆腐とねぎのみそ汁ごはん", "ねぎ", 30, "g"),
    ("R006", "豆腐とねぎのみそ汁ごはん", "みそ", 15, "g"),  # 量は仮置き（要確認）
]

# RECIPE_TIMES：レシピごとの調理時間・片付け時間（分）。
#   いまは仮の値（根拠なし）。実際に作って測るか、根拠を決めて差し替える。
RECIPE_TIMES = {
    # recipe_id: (cook_min, cleanup_min)
    "R001": (20, 10),
    "R002": (20, 10),
    "R003": (30, 10),
    "R004": (15, 10),
    "R005": (10, 5),
    "R006": (20, 10),
}

# INGREDIENT_TO_ITEM：{レシピでの短い食材名: prices.csv の item_id（e-Stat の品目番号）}。
#   ここで食材名と価格データを結びつける。新しい食材を使うときは、まず prices.ipynb で価格を用意する。
INGREDIENT_TO_ITEM = {
    "うるち米": "1001", "ゆでうどん": "1031", "スパゲッティ": "1042", "さけ": "1106",
    "豚肉": "1211", "鶏肉": "1221", "牛乳": "1303", "鶏卵": "1341",
    "キャベツ": "1401", "もやし": "1407", "じゃがいも": "1412", "にんじん": "1415",
    "たまねぎ": "1417", "納豆": "1473", "食用油": "1601", "しょう油": "1621",
    "みそ": "1631", "調理パン": "1792",
    "食パン": "1021", "豆腐": "1471", "トマト": "1436", "ねぎ": "1405",
}

## 関数定義（定義セル。`st.` は使わない／セル間で変数を共有しない）

> **このセル**：量をgにそろえる、レシピ表を作る、検算する、価格を1gあたりに直す、材料費を出す、の関数たち（ここは定義だけ）。
>
> **聞かれたら**：材料費は、食材ごとの「使う量(g)×1gあたりの値段」を足したもの。1gあたりの値段＝価格÷単位の重さ（例：米は4,879円÷5,000g）。

In [ ]:
# ===== 関数定義セル（定義だけ。`st.` は使わない／セル間で変数を共有しない） =====

def to_grams(amount, unit, egg_g_per_piece, milk_g_per_ml):
    """使用量を g（重さ）に換算する。

    入力: amount は数量、unit は "g" / "個"（鶏卵）/ "mL"（牛乳）。
          egg_g_per_piece は卵1個の重さ(g)、milk_g_per_ml は 1mL あたりの重さ(g)。
    戻り値: 重さ（g）の数値。未対応の単位ならエラーにする（黙って間違えないため）。
    """
    if unit == "g":
        return float(amount)                       # もう g なのでそのまま
    if unit == "個":
        return float(amount) * egg_g_per_piece     # 個数 × 1個の重さ
    if unit == "mL":
        return float(amount) * milk_g_per_ml       # mL × 1mLの重さ
    raise ValueError(f"未対応の単位: {unit}")


def build_recipes(raw_recipes, recipe_times, ingredient_to_item, prices_df, egg_g_per_piece, milk_g_per_ml):
    """入力したレシピを recipes.csv の形（team_rules 第2節）に整える。

    入力: raw_recipes は (recipe_id, name, ingredient, amount, unit) のリスト。
          recipe_times は {recipe_id: (cook_min, cleanup_min)}。
          ingredient_to_item は {短い食材名: item_id}。prices_df は prices.csv の DataFrame。
    戻り値: DataFrame（recipe_id, name, ingredient, amount_g, cook_min, cleanup_min）
          ingredient は prices.csv の name と同じ文字列。
    """
    df = pd.DataFrame(raw_recipes, columns=["recipe_id", "name", "short_name", "amount", "unit"])
    # prices.csv から {item_id: 正式な品目名} を作り、短い食材名 → item_id → 正式名 の順に変換する
    item_names = prices_df.drop_duplicates("item_id").set_index("item_id")["name"]
    df["item_id"] = df["short_name"].map(ingredient_to_item)
    df["ingredient"] = df["item_id"].map(item_names)
    # 使用量を g にそろえる（個・mL を換算）
    df["amount_g"] = [to_grams(a, u, egg_g_per_piece, milk_g_per_ml) for a, u in zip(df["amount"], df["unit"])]
    # レシピごとの調理時間・片付け時間を付ける（RECIPE_TIMES に無ければ空欄）
    df["cook_min"] = df["recipe_id"].map(lambda r: recipe_times.get(r, (None, None))[0])
    df["cleanup_min"] = df["recipe_id"].map(lambda r: recipe_times.get(r, (None, None))[1])
    return df[["recipe_id", "name", "ingredient", "amount_g", "cook_min", "cleanup_min"]]


def check_recipes(recipes_df):
    """recipes.csv の検算。問題点の文のリストを返す（空なら問題なし）。

    チェック内容: 価格のない食材／使用量0以下／調理・片付け時間の抜け／同じ食材の重複／料理名のぶれ。
    """
    problems = []
    if recipes_df["ingredient"].isna().any():      # 食材名が価格データと結びつかなかった行
        bad = recipes_df[recipes_df["ingredient"].isna()]["recipe_id"].unique()
        problems.append(f"prices.csv にない食材がある: {list(bad)}")
    if (recipes_df["amount_g"] <= 0).any():        # 使用量が0以下はおかしい
        problems.append("使用量が0以下の行がある")
    if recipes_df[["cook_min", "cleanup_min"]].isna().any().any():   # 時間が入っていない
        bad = recipes_df[recipes_df["cook_min"].isna() | recipes_df["cleanup_min"].isna()]["recipe_id"].unique()
        problems.append(f"調理・片付け時間がないレシピ: {list(bad)}")
    if recipes_df.duplicated(["recipe_id", "ingredient"]).any():     # 同じレシピに同じ食材が2行
        problems.append("同じレシピに同じ食材が2行ある")
    if (recipes_df.groupby("recipe_id")["name"].nunique() > 1).any():  # 1つのIDに複数の料理名
        problems.append("同じ recipe_id に別の料理名がある")
    return problems


def grams_in_unit(unit, egg_g_per_piece, ml_g_per_ml):
    """prices.csv の unit（価格がカバーする量）を g に直す。例: "5kg"->5000, "100g"->100, "10個"->10個ぶん, "450mL"->mL×換算。

    入力: unit は "5kg" "900g" "10個" "1000mL" のような文字列。空欄・未知の形式は NaN を返す。
    """
    import re
    m = re.fullmatch(r"(\d+(?:\.\d+)?)(kg|g|個|mL)", str(unit))   # 「数字＋単位」の形かを判定
    if not m:
        return float("nan")                                       # 単位が空欄・未知なら計算できない
    n, u = float(m.group(1)), m.group(2)
    return {"kg": n * 1000, "g": n, "個": n * egg_g_per_piece, "mL": n * ml_g_per_ml}[u]


def price_per_gram(prices_df, egg_g_per_piece, ml_g_per_ml):
    """prices.csv の価格を 1gあたり（円/g）に換算する。単位が空欄・未知の品目は NaN。

    考え方: 価格は「unit ぶんの値段」なので、価格 ÷ (unit を g にした量) が 1g あたりの値段になる。
    """
    out = prices_df.copy()
    out["grams"] = out["unit"].map(lambda u: grams_in_unit(u, egg_g_per_piece, ml_g_per_ml))
    out["yen_per_g"] = out["price_yen"] / out["grams"]
    return out[["item_id", "name", "month", "yen_per_g"]]


def ingredient_cost(recipes_df, prices_df, month, egg_g_per_piece, milk_g_per_ml):
    """指定した月の、レシピごとの材料費（ロスなし）を計算する。

    計算: 食材ごとに（使用量 g × 1gあたりの円）を出し、レシピごとに合計する。
    戻り値: dict
        {"costs": DataFrame(recipe_id, name, ingredient_yen),
         "missing_price": 単位が未確認で計算に入らなかった食材のリスト}
    """
    ppg = price_per_gram(prices_df, egg_g_per_piece, milk_g_per_ml)
    ppg = ppg[ppg["month"] == month][["name", "yen_per_g"]].rename(columns={"name": "ingredient"})  # 指定月だけ
    detail = recipes_df.merge(ppg, on="ingredient", how="left")        # レシピの各食材に 1g あたりの値段を付ける
    missing = sorted(detail[detail["yen_per_g"].isna()]["ingredient"].unique())   # 値段が付かなかった食材
    detail["yen"] = detail["amount_g"] * detail["yen_per_g"]            # 食材ごとの金額
    costs = detail.groupby(["recipe_id", "name"], as_index=False)["yen"].sum(min_count=1)  # レシピごとに合計
    return {"costs": costs.rename(columns={"yen": "ingredient_yen"}), "missing_price": missing}

## 動作確認セル（定義セルとは分ける）

> **このセル**：レシピ表を作って検算する（6品・26行）。
>
> **聞かれたら**：検算で見ているのは、価格のない食材、量が0以下、時間の抜け、同じ食材の重複、料理名のぶれ。

In [ ]:
# ===== 動作確認セル1：レシピ表を作って検算する =====
# 価格表を読む（item_id は "1001" のような文字列として読む。数値にすると先頭の0などが崩れるため）
prices = pd.read_csv(PRICE_PATH, dtype={"item_id": str})
# 材料表 RAW_RECIPES を recipes.csv の形に整える
recipes = build_recipes(RAW_RECIPES, RECIPE_TIMES, INGREDIENT_TO_ITEM, prices, EGG_G_PER_PIECE, MILK_G_PER_ML)
print("レシピ数:", recipes["recipe_id"].nunique(), "/ 行数:", len(recipes))
print("検算:", check_recipes(recipes) or "問題なし")   # 問題があれば、その内容の文が表示される
recipes                                                  # 表を表示して目で確認する

### 検算: 最新月の材料費（ロスなし・時間コストなし）

> **このセル**：最新月（2026-08）の、レシピごとの材料費を出す。
>
> **聞かれたら**：ロス（捨てる分）と時間のコストはまだ入れていない。

In [ ]:
# ===== 動作確認セル2：最新月の、レシピごとの材料費を出す =====
# （このセルは単独で動くよう、セル1の変数を使わず、ここで読み込み直している）
latest = sorted(pd.read_csv(PRICE_PATH)["month"].unique())[-1]   # prices.csv にある一番新しい月（例 2026-08）
result = ingredient_cost(
    build_recipes(RAW_RECIPES, RECIPE_TIMES, INGREDIENT_TO_ITEM, pd.read_csv(PRICE_PATH, dtype={"item_id": str}),
                  EGG_G_PER_PIECE, MILK_G_PER_ML),
    pd.read_csv(PRICE_PATH, dtype={"item_id": str}), latest, EGG_G_PER_PIECE, MILK_G_PER_ML,
)
print("対象月:", latest)
print("単位が未確認で材料費に入っていない食材:", result["missing_price"] or "なし")
result["costs"].round(0)                                          # レシピごとの材料費（円・1人分・ロスなし）

### 月ごとの自炊費用の推移（ロスなし・時間コストなし）

レシピの量は固定し、食材の値段だけを月ごとに入れ替えて、1食の材料費がどう動いたかを見る（第3回シート「次回に向けた作業」）。

> **このセル**：全部の月の材料費を出す関数、最初と最後の月を比べる関数、推移グラフを描く関数（ここは定義だけ）。
>
> **聞かれたら**：レシピの量は固定して、食材の値段だけを月ごとに入れ替えている。

In [ ]:
# ===== 関数定義セル2：月ごとの材料費と、その推移グラフ（定義だけ） =====

def monthly_cost(recipes_df, prices_df, egg_g_per_piece, milk_g_per_ml):
    """prices.csv にあるすべての月について、レシピごとの材料費（1人分・ロスなし）を計算する。

    考え方: 月ごとに ingredient_cost を呼び、縦に並べる。レシピ（量）は固定で、食材の値段だけが月で変わる。
    戻り値: dict
        {"costs": DataFrame(month, recipe_id, name, ingredient_yen),
         "missing_price": どこかの月で値段が付かなかった食材のリスト}
    """
    frames, missing = [], set()
    for month in sorted(prices_df["month"].unique()):               # 古い月から順に
        r = ingredient_cost(recipes_df, prices_df, month, egg_g_per_piece, milk_g_per_ml)
        frames.append(r["costs"].assign(month=month))
        missing |= set(r["missing_price"])
    costs = pd.concat(frames, ignore_index=True)[["month", "recipe_id", "name", "ingredient_yen"]]
    return {"costs": costs, "missing_price": sorted(missing)}


def cost_change(costs_df):
    """最初の月と最後の月の材料費を比べた表を返す（レシピごと＋6品の平均）。

    戻り値: DataFrame(recipe_id, name, first_yen, last_yen, change_yen, change_pct)
    """
    months = sorted(costs_df["month"].unique())
    first = costs_df[costs_df["month"] == months[0]].set_index("recipe_id")
    last = costs_df[costs_df["month"] == months[-1]].set_index("recipe_id")
    t = pd.DataFrame({"name": first["name"], "first_yen": first["ingredient_yen"], "last_yen": last["ingredient_yen"]})
    t.loc["平均"] = ["全レシピの平均", t["first_yen"].mean(), t["last_yen"].mean()]   # 1食あたりの平均
    t["change_yen"] = t["last_yen"] - t["first_yen"]
    t["change_pct"] = t["change_yen"] / t["first_yen"] * 100
    return t.reset_index()


def plot_cost_trend(costs_df):
    """レシピごとの材料費（円）の推移を折れ線で描く。右端に「料理名 〇円（+〇%）」を直接書く。

    入力: costs_df は monthly_cost の戻り値の costs。
    戻り値: matplotlib の Figure
    """
    from matplotlib import font_manager
    # グラフの日本語が文字化けしないよう、使えるフォントを上から順に探す（prices.ipynb と同じ）
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for fname in ["Meiryo", "Yu Gothic", "MS Gothic", "Noto Sans CJK JP", "IPAexGothic", "IPAGothic"]:
        if fname in installed:
            plt.rcParams["font.family"] = fname
            break

    # 系列の色（データ可視化の基準パレットの1〜6番。sensitivity.ipynb の色と同じ並び）
    colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
    ink, ink2, grid, bg = "#0b0b0b", "#52514e", "#e4e3de", "#fcfcfb"   # 文字色・補助文字色・目盛り線・背景

    table = costs_df.pivot(index="month", columns="recipe_id", values="ingredient_yen")   # 行=月、列=レシピ
    names = costs_df.drop_duplicates("recipe_id").set_index("recipe_id")["name"]
    fig, ax = plt.subplots(figsize=(9.5, 5.4), facecolor=bg)
    ax.set_facecolor(bg)
    x = list(range(len(table)))

    # 右端のラベルが重ならないよう、最終月の値の順に並べて最小の間隔をあける
    order = table.iloc[-1].sort_values().index
    label_y, prev = {}, None
    gap = (table.max().max() - table.min().min()) * 0.06
    for rid in order:
        y = table[rid].iloc[-1]
        if prev is not None and y - prev < gap:
            y = prev + gap
        label_y[rid], prev = y, y

    for color, rid in zip(colors, table.columns):
        ax.plot(x, table[rid], color=color, linewidth=2, label=f"{rid} {names[rid]}")
        end, start = table[rid].iloc[-1], table[rid].iloc[0]
        ax.text(len(table) - 0.7, label_y[rid], f"{names[rid]}  {end:.0f}円（{(end / start - 1) * 100:+.0f}%）",
                color=ink, va="center", fontsize=9)

    ticks = sorted(set([0, 3, 6, 9, len(table) - 1]))
    ax.set_xticks(ticks)
    ax.set_xticklabels([table.index[i] for i in ticks], color=ink2)
    ax.set_xlim(-0.3, len(table) + 4.2)                          # 右に余白を取り、直接ラベルの置き場にする
    ax.set_ylim(0, table.max().max() * 1.2)                      # 金額なので 0 から
    ax.tick_params(axis="y", colors=ink2)
    ax.grid(axis="y", color=grid, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(grid)
    ax.set_ylabel("1食の材料費（円・1人分・ロスなし）", color=ink2)
    ax.set_title("自炊1食の材料費の推移（東京都区部の価格・直近12か月）", loc="left", color=ink, fontsize=13, pad=14)
    ax.legend(loc="upper left", frameon=False, fontsize=8, labelcolor=ink2, ncol=3)
    fig.text(0.01, 0.01, "出典：e-Stat 小売物価統計調査（動向編）を加工して作成。レシピは分析用の仮想レシピ",
             color=ink2, fontsize=8)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    return fig


> **このセル**：12か月×6品の材料費を出して、最新月の値がセル2の結果と一致するか確かめる。
>
> **聞かれたら**：結果：6品の平均は348円→354円（+1.7%）で、ほぼ横ばい。月ごとの値動きより、何を作るか（196〜656円）の差の方が大きい。

In [ ]:
# ===== 動作確認セル3：月ごとの材料費を出し、推移を確認する =====
# （このセルは単独で動くよう、ここで読み込み直している）
prices_m = pd.read_csv(PRICE_PATH, dtype={"item_id": str})
recipes_m = build_recipes(RAW_RECIPES, RECIPE_TIMES, INGREDIENT_TO_ITEM, prices_m, EGG_G_PER_PIECE, MILK_G_PER_ML)
monthly = monthly_cost(recipes_m, prices_m, EGG_G_PER_PIECE, MILK_G_PER_ML)
costs_m = monthly["costs"]
print("月数:", costs_m["month"].nunique(), "/ レシピ数:", costs_m["recipe_id"].nunique(), "/ 行数:", len(costs_m))
print("値段が付かなかった食材:", monthly["missing_price"] or "なし")

# 検算：最新月の値が、動作確認セル2（ingredient_cost を最新月で直接呼んだ結果）と一致するか
latest_m = sorted(costs_m["month"].unique())[-1]
direct = ingredient_cost(recipes_m, prices_m, latest_m, EGG_G_PER_PIECE, MILK_G_PER_ML)["costs"].set_index("recipe_id")
via = costs_m[costs_m["month"] == latest_m].set_index("recipe_id")
assert (direct["ingredient_yen"] - via["ingredient_yen"]).abs().max() < 1e-9, "月ごとの計算と最新月の計算が合わない"
print("検算: 最新月の値が動作確認セル2と一致")

# 最初の月 → 最新月 の変化（レシピごと＋平均）
cost_change(costs_m).round(1)


> **このセル**：推移グラフを `docs/cooking_cost_trend.png` に保存する。
>
> **聞かれたら**：金額のグラフなので、差を大げさに見せないよう縦軸を0円から始めている。

In [ ]:
# ===== 動作確認セル4：推移グラフを描いて保存する =====
fig = plot_cost_trend(costs_m)
fig.savefig(OUT_FIG_PATH, dpi=150)
print("保存しました:", OUT_FIG_PATH)


## 確認メモ（手で埋める）

- [ ] 調理時間・片付け時間は仮の値。実測か根拠を決めて差し替える（team_rules 第8節の未決事項）
- [ ] 鶏卵 1個=60g、牛乳 1mL=1g は仮定。`src/config.py` ができたらそちらに移す
- [ ] R003 のみそ 15g は料理名に合わせて追加した。量を確認する
- [x] 価格の単位は 22品目すべて確認済み（`data/item_map.csv`）。材料費に入らない食材はなくなった
- [ ] 他のメンバーのレシピを受け取ったら、同じ書式で `RAW_RECIPES` に追記し、検算セルで確認する
- [x] 栄養（kcal・たんぱく質・脂質・炭水化物）は `nutrition.ipynb` で成分表から付けた（利用条件も確認済み）
- [x] 月ごとの自炊費用の推移（`cooking_cost_monthly.csv`・`cooking_cost_trend.png`）
- [ ] 自炊と外食の費用差の比較（外食データ待ち）。前の版にあった外食との比較（外食価格は仮の値）は外した。外食は王さんの担当（`prices_out.ipynb`）

## 保存

> **このセル**：`recipes.csv` と、月ごとの材料費 `cooking_cost_monthly.csv` を保存する。
>
> **聞かれたら**：`recipes.csv` の列は team_rules 第2節どおり。

In [ ]:
# ===== 保存セル：完成したレシピ表を data/recipes.csv に保存する（チームの共通データ） =====
recipes.to_csv(OUT_PATH, index=False, encoding="utf-8")   # index=False は行番号を書き出さない指定
print("保存しました:", OUT_PATH)

# 月ごとの材料費（動作確認セル3で作ったもの）も保存する
costs_m.to_csv(OUT_COST_PATH, index=False, encoding="utf-8")
print("保存しました:", OUT_COST_PATH)